## Exercise 03. Aggregations
---


In [1]:
import pandas as pd 
import sqlite3

conn = sqlite3.connect('data/checking-logs.sqlite')

In [2]:
schema = pd.io.sql.read_sql("PRAGMA table_info(test);", conn)

In [3]:
first_rows = pd.io.sql.read_sql("SELECT * FROM test LIMIT 10;", conn)
print(first_rows)

       uid   labname             first_commit_ts               first_view_ts
0   user_1    laba04  2020-04-26 17:06:18.462708  2020-04-26 21:53:59.624136
1   user_1   laba04s  2020-04-26 17:12:11.843671  2020-04-26 21:53:59.624136
2   user_1    laba05  2020-05-02 19:15:18.540185  2020-04-26 21:53:59.624136
3   user_1    laba06  2020-05-17 16:26:35.268534  2020-04-26 21:53:59.624136
4   user_1   laba06s  2020-05-20 12:23:37.289724  2020-04-26 21:53:59.624136
5   user_1  project1  2020-05-14 20:56:08.898880  2020-04-26 21:53:59.624136
6  user_10    laba04  2020-04-25 08:24:52.696624  2020-04-18 12:19:50.182714
7  user_10   laba04s  2020-04-25 08:37:54.604222  2020-04-18 12:19:50.182714
8  user_10    laba05  2020-05-01 19:27:26.063245  2020-04-18 12:19:50.182714
9  user_10    laba06  2020-05-19 11:39:28.885637  2020-04-18 12:19:50.182714


In [4]:
schema_dead = pd.io.sql.read_sql("PRAGMA table_info(deadlines);", conn)
print(schema_dead)

   cid       name     type  notnull dflt_value  pk
0    0      index  INTEGER        0       None   0
1    1       labs     TEXT        0       None   0
2    2  deadlines  INTEGER        0       None   0


In [5]:
query_min = """
SELECT 
    t.uid,
    ((CAST(strftime('%s', t.first_commit_ts) AS INTEGER) - d.deadlines) / 3600) AS delta_hours_min
FROM test t 
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
GROUP BY t.uid,t.labname
ORDER BY delta_hours_min ASC
LIMIT 1
"""

df_min = pd.io.sql.read_sql(query_min, conn)
df_min

,uid,delta_hours_min
0,user_30,-202


In [6]:
query_max = """
SELECT 
    t.uid,
    ((CAST(strftime('%s', t.first_commit_ts) AS INTEGER) - d.deadlines) / 3600) AS delta_hours_max
FROM test t 
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
GROUP BY t.uid,t.labname
ORDER BY delta_hours_max DESC
LIMIT 1
"""

df_max = pd.io.sql.read_sql(query_max, conn)
df_max

,uid,delta_hours_max
0,user_25,-2


In [7]:
query_avg = """
SELECT
    AVG((CAST(strftime('%s', t.first_commit_ts) AS INTEGER) - d.deadlines) / 3600) AS avg_delta_hours
FROM test t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
"""

df_avg = pd.io.sql.read_sql(query_avg, conn)
df_avg

,avg_delta_hours
0,-89.125


In [8]:
conn.execute("DROP TABLE IF EXISTS views_diff")
conn.commit()

query_corr = """
CREATE TABLE views_diff AS
SELECT 
    t.uid,
    AVG((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600) AS avg_diff,
    COUNT(p.datetime) AS pageviews
FROM test t
JOIN deadlines d ON t.labname = d.labs
LEFT JOIN pageviews p ON t.uid = p.uid
WHERE t.labname != 'project1'
GROUP BY t.labname, t.uid
"""
conn.execute(query_corr)
conn.commit()

views_diff = pd.io.sql.read_sql("SELECT * FROM views_diff", conn)
views_diff[['avg_diff', 'pageviews']].corr()

,avg_diff,pageviews
avg_diff,1.000000,-0.149382
pageviews,-0.149382,1.000000


In [9]:
conn.close()